In [1]:
from pathlib import Path
import json
import math
import re
import statistics

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, GPT2LMHeadModel

## path and device

In [2]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = project_root / "data"
outputs_dir = project_root / "outputs"

model_dir = outputs_dir / "punjabi-gpt-scratch-30m-final"
valid_jsonl = data_dir / "punjabi_corpus_cleaned.jsonl"   # we will sample from this

device = "mps" if torch.backends.mps.is_available() else "cpu"

print("Model dir:", model_dir)
print("Validation file:", valid_jsonl)
print("Device:", device)

Model dir: /Users/gurpejsingh/Desktop/gpt2-finetune-mac/training-punjabi-gpt/outputs/punjabi-gpt-scratch-30m-final
Validation file: /Users/gurpejsingh/Desktop/gpt2-finetune-mac/training-punjabi-gpt/data/punjabi_corpus_cleaned.jsonl
Device: mps


## load model and tokenizer

In [3]:
tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
model = GPT2LMHeadModel.from_pretrained(str(model_dir)).to(device)
model.eval()

print("Parameter count:", f"{model.num_parameters():,}")
print("Tokenizer vocab size:", len(tokenizer))
print("Context length:", model.config.n_positions)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Parameter count: 20,389,632
Tokenizer vocab size: 16000
Context length: 128


## helper function

In [4]:
def is_gurmukhi_char(ch: str) -> bool:
    return "\u0A00" <= ch <= "\u0A7F"

def gurmukhi_ratio(text: str) -> float:
    chars = [ch for ch in text if not ch.isspace()]
    if not chars:
        return 0.0
    return sum(1 for ch in chars if is_gurmukhi_char(ch)) / len(chars)

def repetition_rate_words(text: str, n=3) -> float:
    words = text.split()
    if len(words) < n:
        return 0.0
    ngrams = [" ".join(words[i:i+n]) for i in range(len(words)-n+1)]
    if not ngrams:
        return 0.0
    unique = len(set(ngrams))
    total = len(ngrams)
    return 1 - (unique / total)

def generate_text(prompt, max_new_tokens=100, temperature=0.8, top_p=0.95):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## fixed prompt benchmark set

In [5]:
benchmark_prompts = [
    "ਪੰਜਾਬੀ ਭਾਸ਼ਾ",
    "ਇੱਕ ਪਿੰਡ ਵਿੱਚ",
    "ਅੱਜ ਦੇ ਸਮੇਂ ਵਿੱਚ ਸਿੱਖਿਆ",
    "ਕਿਸਾਨ ਲਈ ਸਭ ਤੋਂ ਮਹੱਤਵਪੂਰਨ ਗੱਲ",
    "ਪੰਜਾਬ ਦੇ ਇਤਿਹਾਸ ਬਾਰੇ",
    "ਪਰਿਵਾਰ ਦੀ ਮਹੱਤਤਾ",
    "ਸਿਹਤਮੰਦ ਜੀਵਨ ਸ਼ੈਲੀ",
    "ਇੱਕ ਛੋਟੀ ਕਹਾਣੀ",
    "ਤਕਨਾਲੋਜੀ ਦਾ ਭਵਿੱਖ",
    "ਬੱਚਿਆਂ ਦੀ ਸਿੱਖਿਆ"
]

print("Prompt count:", len(benchmark_prompts))

Prompt count: 10


## run prompt benchmark

In [6]:
generation_results = []

for prompt in benchmark_prompts:
    output = generate_text(prompt, max_new_tokens=120)
    generation_results.append({
        "prompt": prompt,
        "output": output,
        "gurmukhi_ratio": gurmukhi_ratio(output),
        "repetition_rate_3gram_words": repetition_rate_words(output, n=3),
        "char_length": len(output),
        "word_length": len(output.split())
    })

for item in generation_results[:3]:
    print("=" * 80)
    print("PROMPT:", item["prompt"])
    print("OUTPUT:", item["output"])

This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (128). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


PROMPT: ਪੰਜਾਬੀ ਭਾਸ਼ਾ
OUTPUT: ਪੰਜਾਬੀ ਭਾਸ਼ਾ

ਪੰਜਾਬੀ ਭਾਸ਼ਾ ਕੋਰੀਆ ਦੇ ਬੋਹੀਮੀਅਨ ਰਾਜ ਵਿੱਚ ਇੱਕ ਪ੍ਰਾਚੀਨ ਪੱਥਰ ਦਾ ਭਾਸ਼ਾ ਹੈ। ਇਹ ਪੱਥਰ ਦੇ ਨੇਡ਼ੇ ਇੱਕ ਪ੍ਰਤੱਖ ਭੂਮੀਗਤ ਮੈਂਬਰ ਹੈ ਜਿਸ ਦਾ ਵਰਣਨ ਪਹਿਲੀ ਵਾਰ 2011 ਵਿੱਚ ਨਿਕੋਲਾ ਕਾਮਿਨ ਦੁਆਰਾ ਕੀਤਾ ਗਿਆ
PROMPT: ਇੱਕ ਪਿੰਡ ਵਿੱਚ
OUTPUT: ਇੱਕ ਪਿੰਡ ਵਿੱਚ ਸੂਚੀਬੱਧ ਇਮਾਰਤਾਂ

ਇੱਕ ਪਿੰਡ ਵਿੱਚ ਸੂਚੀਬੱਧ ਇਮਾਰਤਾਂ ਇੱਕ ਇਤਿਹਾਸਕ ਇਮਾਰਤ ਹੈ ਜੋ ਇੱਕ ਗ੍ਰੇਡ II ਸੂਚੀਬੱਧ ਇਮਾਰਤ ਵਜੋਂ ਸੂਚੀਬੱਧ ਹੈ। ਇਮਾਰਤ ਨੂੰ ਇੱਕ ਸਿਵਲ ਪੈਰੀਸ਼ ਦੇ ਰੂਪ ਵਿੱਚ ਸੂਚੀਬੱਧ ਇਮਾਰਤ ਵਜੋਂ ਦਰਜ ਕੀਤਾ
PROMPT: ਅੱਜ ਦੇ ਸਮੇਂ ਵਿੱਚ ਸਿੱਖਿਆ
OUTPUT: ਅੱਜ ਦੇ ਸਮੇਂ ਵਿੱਚ ਸਿੱਖਿਆ

ਅੱਜ ਦੇ ਸਮੇਂ ਵਿੱਚ ਸਿੱਖਿਆ (ਜਿਸ ਨੂੰ ਅੱਜ ਦੇ ਸਮੇਂ ਵੀ ਕਿਹਾ ਜਾਂਦਾ ਹੈ) ਇੱਕ ਵਿਦੇਸ਼ੀ ਕੰਪਲੈਕਸ ਹੈ ਜੋ ਇੱਕ ਸਿੱਖਿਆ ਦੀ ਖੋਜ ਅਤੇ ਵਿਸ਼ੇਸ਼ਤਾ ਲਈ ਅੱਜ ਦੇ ਸਮੇਂ ਵਿੱਚ ਸਿੱਖਿਆ ਦੀ ਪੇਸ਼ਕਸ਼ ਕਰਦਾ ਹੈ। ਅੱਜ, ਜੋਰ, ਵ


## generate summary metrices

In [7]:
avg_gurmukhi_ratio = statistics.mean(x["gurmukhi_ratio"] for x in generation_results)
avg_repetition = statistics.mean(x["repetition_rate_3gram_words"] for x in generation_results)
avg_char_length = statistics.mean(x["char_length"] for x in generation_results)
avg_word_length = statistics.mean(x["word_length"] for x in generation_results)

print("Average Gurmukhi ratio:", round(avg_gurmukhi_ratio, 4))
print("Average repetition rate (3-gram words):", round(avg_repetition, 4))
print("Average char length:", round(avg_char_length, 2))
print("Average word length:", round(avg_word_length, 2))

Average Gurmukhi ratio: 0.9466
Average repetition rate (3-gram words): 0.0481
Average char length: 198.7
Average word length: 38


## preplexity evaluation

In [8]:
eval_dataset = load_dataset(
    "json",
    data_files={"eval": str(valid_jsonl)}
)["eval"]

# Create a small held-out evaluation slice for reproducibility
eval_subset = eval_dataset.select(range(min(500, len(eval_dataset))))
print("Held-out eval rows:", len(eval_subset))

Generating eval split: 0 examples [00:00, ? examples/s]

Held-out eval rows: 500


## build a long evaluation text

In [9]:
eval_text = "\n\n".join([row["text"] for row in eval_subset if row["text"].strip()])

print("Eval text characters:", len(eval_text))
print("Eval text preview:", eval_text[:500])

Eval text characters: 1388634
Eval text preview: 1891 ਸੰਯੁਕਤ ਰਾਜ ਅਮਰੀਕਾ ਦੀ ਸੀਨੇਟ ਚੋਣਾਂ

ਸੰਯੁਕਤ ਰਾਜ ਅਮਰੀਕਾ ਦੀ ਸੈਨੇਟ ਚੋਣਾਂ 20 ਅਤੇ 21 ਜਨਵਰੀ 1891 ਨੂੰ ਨਿਊਯਾਰਕ ਰਾਜ ਵਿਧਾਨ ਸਭਾ ਦੁਆਰਾ ਇੱਕ ਯੂ ਦੀ ਚੋਣ ਕਰਨ ਲਈ ਆਯੋਜਿਤ ਕੀਤੀਆਂ ਗਈਆਂ ਸਨ।ਐਸ. ਸੈਨੇਟਰ (ਕਲਾਸ 3), ਸੰਯੁਕਤ ਰਾਜ ਦੀ ਸੀਨੇਟ ਵਿੱਚ ਨਿਊਯਾਰਕ ਰਾਜ ਦੀ ਨੁਮਾਇੰਦਗੀ ਕਰਨ ਲਈ।
ਪਿਛੋਕਡ਼.
ਰਿਪਬਲਿਕਨ ਵਿਲੀਅਮ ਐਮ. ਈਵਾਰਟਸ 1885 ਵਿੱਚ ਇਸ ਸੀਟ ਲਈ ਚੁਣੇ ਗਏ ਸਨ ਅਤੇ ਉਨ੍ਹਾਂ ਦਾ ਕਾਰਜਕਾਲ 3 ਮਾਰਚ 1891 ਨੂੰ ਖਤਮ ਹੋ ਰਿਹਾ ਸੀ।
ਨਵੰਬਰ 1889 ਵਿੱਚ ਰਾਜ ਦੀਆਂ ਚੋਣਾਂ ਵਿੱਚ, ਰਾਜ ਦੀ ਸੀਨੇਟ ਵਿੱਚ 19 ਗਣਤੰਤਰ ਅਤੇ 13 ਲੋਕਤੰਤਰੀ ਦੋ ਸਾਲ ਦੇ ਕਾਰਜਕਾਲ (1890-1891) ਲਈ ਚੁਣੇ ਗਏ ਸਨ। ਨਵੰਬਰ 


## sliding window perplexity

In [10]:
encodings = tokenizer(eval_text, return_tensors="pt")
seq_len = encodings.input_ids.size(1)

max_length = model.config.n_positions
stride = 64

nlls = []
prev_end_loc = 0

for begin_loc in range(0, seq_len, stride):
    end_loc = min(begin_loc + max_length, seq_len)
    trg_len = end_loc - prev_end_loc

    input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
    target_ids = input_ids.clone()
    target_ids[:, :-trg_len] = -100

    with torch.no_grad():
        outputs = model(input_ids, labels=target_ids)
        neg_log_likelihood = outputs.loss

    nlls.append(neg_log_likelihood)
    prev_end_loc = end_loc

    if end_loc == seq_len:
        break

ppl = torch.exp(torch.stack(nlls).mean()).item()
eval_loss_est = torch.stack(nlls).mean().item()

print("Estimated eval loss:", eval_loss_est)
print("Perplexity:", ppl)

Token indices sequence length is longer than the specified maximum sequence length for this model (872007 > 1024). Running this sequence through the model will result in indexing errors
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Estimated eval loss: 1.9491775035858154
Perplexity: 7.022909164428711


## combine benchmark report

In [11]:
benchmark_report = {
    "model_name": "punjabi-gpt-scratch-30m",
    "model_type": "decoder-only causal language model",
    "language": "Punjabi (Gurmukhi)",
    "parameter_count": int(model.num_parameters()),
    "tokenizer_vocab_size": int(len(tokenizer)),
    "context_length": int(model.config.n_positions),
    "device": device,
    "intrinsic_metrics": {
        "eval_loss_estimate": eval_loss_est,
        "perplexity": ppl
    },
    "generation_metrics": {
        "avg_gurmukhi_ratio": avg_gurmukhi_ratio,
        "avg_repetition_rate_3gram_words": avg_repetition,
        "avg_generated_char_length": avg_char_length,
        "avg_generated_word_length": avg_word_length
    },
    "prompt_results": generation_results
}

benchmark_path = outputs_dir / "benchmark_report.json"
with open(benchmark_path, "w", encoding="utf-8") as f:
    json.dump(benchmark_report, f, ensure_ascii=False, indent=2)

print("Saved benchmark report to:", benchmark_path)

Saved benchmark report to: /Users/gurpejsingh/Desktop/gpt2-finetune-mac/training-punjabi-gpt/outputs/benchmark_report.json


##  save plain text sample generation

In [12]:
samples_txt = outputs_dir / "benchmark_samples.txt"

with open(samples_txt, "w", encoding="utf-8") as f:
    for item in generation_results:
        f.write("=" * 100 + "\n")
        f.write(f"PROMPT: {item['prompt']}\n\n")
        f.write(f"OUTPUT:\n{item['output']}\n\n")
        f.write(f"Gurmukhi ratio: {item['gurmukhi_ratio']:.4f}\n")
        f.write(f"Repetition rate (3-gram words): {item['repetition_rate_3gram_words']:.4f}\n")
        f.write(f"Char length: {item['char_length']}\n")
        f.write(f"Word length: {item['word_length']}\n")

print("Saved plain text samples to:", samples_txt)

Saved plain text samples to: /Users/gurpejsingh/Desktop/gpt2-finetune-mac/training-punjabi-gpt/outputs/benchmark_samples.txt
